# Conhecendo o Spark por dentro

No vídeo 1.3, o cálculo de gols por Copa foi dividido entre três computadores. Agora vamos ver quem faz o quê
dentro do Spark.

## Driver e executors

| Peça | O que faz |
|---|---|
| **Driver** | roda o seu programa (aqui, o notebook), monta o plano e distribui o trabalho. O resultado volta para ele |
| **Executor** | faz as contas: lê a sua parte do dado, carrega na memória e calcula |
| **Partição** | cada pedaço em que o dado é dividido |
| **Task** | o trabalho sobre uma partição. Cada núcleo roda uma task por vez |

```
                 DRIVER (o seu programa)
        planeja, cria as tasks, recebe o resultado
          /                  |                  \
  WORKER (máquina)    WORKER (máquina)    WORKER (máquina)
     EXECUTOR            EXECUTOR            EXECUTOR
   núcleo: task        núcleo: task        núcleo: task
   núcleo: task        núcleo: task        núcleo: task
```

Num **cluster**, as máquinas que rodam executors se chamam **workers**. Os três computadores do vídeo 1.3 são os
executors. A fase de juntar acontece entre eles, e só o resultado final volta ao driver.

## E aqui no curso?

Aqui tudo roda no seu computador, dentro do container: é o **modo local**. O driver e o executor ficam no mesmo
processo, e cada **núcleo** roda uma task por vez. É o que o `local[*]` da sessão pede.

```
            container (o seu computador)
   +---------------------------------------------+
   |   DRIVER + EXECUTOR, num processo só        |
   |   núcleo 1     núcleo 2     núcleo 3   ...  |
   |   task         task         task            |
   +---------------------------------------------+
```

**O código é o mesmo**: para levar o programa a um cluster, muda só o `master` da sessão.

## A SparkSession

A `SparkSession` é o **ponto de entrada** do Spark: tudo o que você pede ao Spark passa por ela. Ao criar a sessão,
nasce o driver.

- **`.appName("arena-conhecendo-o-spark")`**: o nome da aplicação. Aparece na Spark UI e nos logs.
- **`.master("local[*]")`**: onde o Spark roda. `local[*]` é o modo local com todos os núcleos; num cluster, entra o
  endereço dele (como `yarn` ou `spark://host:porta`).
- **`.config("spark.sql.warehouse.dir", ...)`**: a pasta das tabelas que o Spark gerencia (aula 3). Só vale se for
  definida antes de a sessão existir.
- **`.getOrCreate()`**: cria a sessão, ou devolve a que já está rodando.

A última linha, `spark`, mostra um resumo da sessão com a versão e o link da Spark UI.

O `defaultParallelism` é quantas tasks o Spark roda **ao mesmo tempo**. Aqui, são os núcleos do container.

## A Spark UI

O driver mantém uma página de acompanhamento no navegador, a **Spark UI**, enquanto a sessão estiver ativa. Cada
notebook tem a sua sessão: a primeira usa a porta **4040**, a seguinte a 4041, e assim por diante. Endereço desta
sessão:

O Spark mostra o nome interno do container, que o navegador não conhece. As portas do container estão ligadas às do
seu computador, então basta usar `localhost`.

A Spark UI organiza o trabalho em três níveis: **job** (um pedido de resultado), **stage** (as etapas do job) e
**task** (uma por partição).

| Aba | O que mostra |
|---|---|
| **Jobs** | cada job, com o tempo de cada um |
| **Stages** | as etapas de cada job e as tasks de cada etapa |
| **Storage** | os dados guardados em cache |
| **Environment** | as configurações da sessão |
| **Executors** | os processos que rodam as tasks |
| **SQL / DataFrame** | o plano de cada consulta, desenhado |

Na aba **Executors**, o modo local mostra uma linha só, com o ID `driver`: driver e executor são o mesmo processo.
Num cluster, aparece também uma linha para cada executor.

## Recapitulando

O **driver** planeja e recebe o resultado, os **executors** calculam, e cada núcleo roda uma task por vez. No modo
local, driver e executor ficam no mesmo processo. As abas **Jobs** e **Stages** ainda estão vazias: nenhum trabalho
foi pedido ao Spark.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()